# ResNet-34 Explainable AI Evaluation with Grad-CAM++

The notebook uses the ResNet-34 model, following the steps:

1. Import libraries  
2. Load dataset samples and test split  
3. Load trained `ResNet-34` model  
4. Define Grad-CAM++ function  
5. Generate Grad-CAM++ heatmaps  
6. Consistency evaluation: original vs perturbed image using SSIM, MSE, IoU  
7. Plausibility evaluation: Grad-CAM vs EBHI-SEG mask using IoU  
8. Fidelity evaluation: deletion test and optional model randomization


## 1. Import libraries

In [9]:
import gc
import json
import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image
from skimage.metrics import structural_similarity as ssim
from skimage.transform import resize as sk_resize
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

In [ ]:
# Configuration
PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.append(str(PROJECT_DIR))

from utils.data_utils import collect_samples, get_eval_transform, stratified_split

DATA_DIR = Path("../data/EBHI-SEG")
OUTPUT_DIR = Path("outputs/xai_evaluation_10_runs")

MODEL_PATHS = {
    42: Path("outputs/resnet34/run_cache/run_0_seed_42_model.pth"),
    123: Path("outputs/resnet34/run_cache/run_1_seed_123_model.pth"),
    2024: Path("outputs/resnet34/run_cache/run_2_seed_2024_model.pth"),
    3407: Path("outputs/resnet34/run_cache/run_3_seed_3407_model.pth"),
    9999: Path("outputs/resnet34/run_cache/run_4_seed_9999_model.pth"),
    7: Path("outputs/resnet34/run_cache/run_5_seed_7_model.pth"),
    77: Path("outputs/resnet34/run_cache/run_6_seed_77_model.pth"),
    314: Path("outputs/resnet34/run_cache/run_7_seed_314_model.pth"),
    2025: Path("outputs/resnet34/run_cache/run_8_seed_2025_model.pth"),
    8888: Path("outputs/resnet34/run_cache/run_9_seed_8888_model.pth")
}

# Use fixed evaluation seed
EVALUATION_SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 1
NUM_WORKERS = 0
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
SAMPLE_LIMIT = None

HEATMAP_THRESHOLD = 0.50
CONSISTENCY_PERTURBATIONS = ["hflip", "vflip", "rotate_10", "rotate_minus_10", "noise"]
DELETION_RATIOS = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Reproducibility and dataset
def set_evaluation_seed(seed=EVALUATION_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class XAITestDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]
        image = Image.open(image_path).convert("RGB")
        tensor = self.transform(image) if self.transform else transforms.ToTensor()(image)
        return tensor, int(label), str(image_path)


def prepare_test_data():
    set_evaluation_seed()
    class_names, class_to_index, samples = collect_samples(str(DATA_DIR))
    train_samples, val_samples, test_samples = stratified_split(
        samples,
        train_ratio=TRAIN_RATIO,
        val_ratio=VAL_RATIO,
        test_ratio=TEST_RATIO,
        seed=EVALUATION_SEED,
    )
    if SAMPLE_LIMIT is not None:
        test_samples = test_samples[:SAMPLE_LIMIT]
    if not test_samples:
        raise ValueError(f"No test images found under {DATA_DIR}")

    dataset = XAITestDataset(test_samples, get_eval_transform(IMAGE_SIZE))
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    print("Device:", DEVICE)
    print("Classes:", class_names)
    print("Total/train/validation/test:", len(samples), len(train_samples), len(val_samples), len(test_samples))
    return class_names, class_to_index, test_samples, loader

In [ ]:
# Model and checkpoint loading
def build_resnet34(num_classes):
    model = models.resnet34(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(512, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, num_classes),
    )
    return model


def load_checkpoint(path, num_classes):
    checkpoint = torch.load(path, map_location=DEVICE)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint
    state_dict = {
        key.removeprefix("module."): value for key, value in state_dict.items()
    }
    model = build_resnet34(num_classes)
    model.load_state_dict(state_dict)
    model.to(DEVICE).eval()
    return model

In [ ]:
# Grad-CAM++
class GradCAMPlusPlus:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self.forward_handle = target_layer.register_forward_hook(self._activation_hook)
        self.backward_handle = target_layer.register_full_backward_hook(self._gradient_hook)

    def _activation_hook(self, module, inputs, output):
        self.activations = output.detach()

    def _gradient_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def remove_hooks(self):
        self.forward_handle.remove()
        self.backward_handle.remove()

    def __call__(self, input_tensor, target_class=None):
        self.model.zero_grad(set_to_none=True)
        output = self.model(input_tensor)
        if hasattr(output, "logits"):
            output = output.logits
        probabilities = torch.softmax(output, dim=1)
        predicted_class = int(probabilities.argmax(dim=1).item())
        predicted_confidence = float(probabilities[0, predicted_class].detach().cpu())
        if target_class is None:
            target_class = predicted_class
        output[0, target_class].backward()

        gradients = self.gradients[0]
        activations = self.activations[0]
        gradients_2 = gradients.pow(2)
        gradients_3 = gradients_2 * gradients
        denominator = 2 * gradients_2 + torch.sum(
            activations * gradients_3, dim=(1, 2), keepdim=True
        )
        alpha = gradients_2 / (denominator + 1e-8)
        weights = torch.sum(alpha * F.relu(gradients), dim=(1, 2), keepdim=True)
        cam = F.relu(torch.sum(weights * activations, dim=0)).detach().cpu().numpy()
        if cam.max() - cam.min() < 1e-8:
            cam = np.zeros_like(cam)
        else:
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        cam = sk_resize(
            cam,
            (input_tensor.shape[2], input_tensor.shape[3]),
            preserve_range=True,
            anti_aliasing=True,
        ).astype(np.float32)
        return cam, predicted_class, predicted_confidence

In [ ]:
# Helpers for calculating the metrics
def heatmap_to_binary(heatmap, threshold=HEATMAP_THRESHOLD):
    heatmap = np.asarray(heatmap, dtype=np.float32)
    if heatmap.max() > 1:
        heatmap = heatmap / heatmap.max()
    return (heatmap >= threshold).astype(np.uint8)


def compute_mse(first, second):
    return float(np.mean((first.astype(np.float32) - second.astype(np.float32)) ** 2))


def compute_ssim(first, second):
    return float(ssim(first.astype(np.float32), second.astype(np.float32), data_range=1.0))


def compute_iou(first, second):
    first, second = first.astype(bool), second.astype(bool)
    union = np.logical_or(first, second).sum()
    if union == 0:
        return np.nan
    return float(np.logical_and(first, second).sum() / union)


def apply_perturbation(input_tensor, name):
    image = input_tensor.clone()
    if name == "hflip":
        return torch.flip(image, dims=[3]), lambda h: np.fliplr(h)
    if name == "vflip":
        return torch.flip(image, dims=[2]), lambda h: np.flipud(h)
    if name == "rotate_10":
        return TF.rotate(image, 10), lambda h: np.asarray(
            Image.fromarray((h * 255).astype(np.uint8)).rotate(-10, resample=Image.BILINEAR),
            dtype=np.float32,
        ) / 255.0
    if name == "rotate_minus_10":
        return TF.rotate(image, -10), lambda h: np.asarray(
            Image.fromarray((h * 255).astype(np.uint8)).rotate(10, resample=Image.BILINEAR),
            dtype=np.float32,
        ) / 255.0
    if name == "noise":
        return image + torch.randn_like(image) * 0.03, lambda h: h
    raise ValueError(f"Unknown perturbation: {name}")


def get_model_confidence(model, input_tensor, target_class):
    with torch.no_grad():
        output = model(input_tensor)
        if hasattr(output, "logits"):
            output = output.logits
        return float(torch.softmax(output, dim=1)[0, target_class].detach().cpu())


def delete_top_heatmap_regions(input_tensor, heatmap, deletion_ratio):
    deleted = input_tensor.clone()
    threshold = np.quantile(heatmap, 1.0 - deletion_ratio)
    mask = torch.as_tensor(heatmap >= threshold, dtype=torch.bool, device=input_tensor.device)
    deleted[:, :, mask] = 0.0
    return deleted


def find_mask_path(image_path):
    image_path = Path(image_path)
    class_dir = image_path.parent.parent if image_path.parent.name.lower() == "image" else image_path.parent
    directories = ["mask", "masks", "label", "labels", "segmentation", "segmentation mask", "segmentation_mask"]
    stems = [image_path.stem, image_path.stem + "_mask", image_path.stem + "_label", image_path.stem + "_seg"]
    extensions = [image_path.suffix, ".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]
    for directory in directories:
        mask_dir = class_dir / directory
        if not mask_dir.exists():
            continue
        for stem in stems:
            for extension in extensions:
                candidate = mask_dir / f"{stem}{extension}"
                if candidate.exists():
                    return candidate
    return None


def load_binary_mask(mask_path):
    mask = Image.open(mask_path).convert("L").resize((IMAGE_SIZE, IMAGE_SIZE), Image.NEAREST)
    return (np.asarray(mask) > 0).astype(np.uint8)


In [ ]:
# Evaluate one checkpoint
def evaluate_checkpoint(model_seed, model_path, num_classes, test_loader):
    print("\n" + "=" * 72)
    print(f"MODEL SEED {model_seed}: {model_path}")
    print("=" * 72)
    set_evaluation_seed()  # identical stochastic perturbations across checkpoints
    run_dir = OUTPUT_DIR / f"seed_{model_seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    model = load_checkpoint(model_path, num_classes)
    gradcam = GradCAMPlusPlus(model, model.layer4[-1])
    heatmap_cache = {}
    heatmap_records = []

    for index, (images, labels, paths) in enumerate(test_loader):
        image, image_path = images.to(DEVICE), paths[0]
        heatmap, predicted_class, predicted_confidence = gradcam(image)
        heatmap_cache[image_path] = {
            "heatmap": heatmap,
            "predicted_class": predicted_class,
            "predicted_confidence": predicted_confidence,
        }
        heatmap_records.append({
            "seed": model_seed,
            "index": index,
            "image_path": image_path,
            "true_label": int(labels.item()),
            "predicted_label": predicted_class,
            "predicted_confidence": predicted_confidence,
        })
        if (index + 1) % 25 == 0 or index + 1 == len(test_loader):
            print(f"Heatmaps: {index + 1}/{len(test_loader)}")

    consistency_records = []
    for index, (images, labels, paths) in enumerate(test_loader):
        image, image_path = images.to(DEVICE), paths[0]
        original = heatmap_cache[image_path]["heatmap"]
        original_binary = heatmap_to_binary(original)
        target_class = heatmap_cache[image_path]["predicted_class"]
        for perturbation in CONSISTENCY_PERTURBATIONS:
            perturbed_image, align_back = apply_perturbation(image, perturbation)
            perturbed, _, _ = gradcam(perturbed_image, target_class=target_class)
            aligned = sk_resize(
                align_back(perturbed), original.shape, preserve_range=True, anti_aliasing=True
            ).astype(np.float32)
            consistency_records.append({
                "seed": model_seed,
                "index": index,
                "image_path": image_path,
                "perturbation": perturbation,
                "ssim": compute_ssim(original, aligned),
                "mse": compute_mse(original, aligned),
                "iou": compute_iou(original_binary, heatmap_to_binary(aligned)),
            })
        if (index + 1) % 25 == 0 or index + 1 == len(test_loader):
            print(f"Consistency: {index + 1}/{len(test_loader)}")

    plausibility_records = []
    for index, (_, labels, paths) in enumerate(test_loader):
        image_path = paths[0]
        mask_path = find_mask_path(image_path)
        value = np.nan if mask_path is None else compute_iou(
            heatmap_to_binary(heatmap_cache[image_path]["heatmap"]), load_binary_mask(mask_path)
        )
        plausibility_records.append({
            "seed": model_seed,
            "index": index,
            "image_path": image_path,
            "true_label": int(labels.item()),
            "mask_path": None if mask_path is None else str(mask_path),
            "mask_found": mask_path is not None,
            "gradcam_mask_iou": value,
        })

    fidelity_records = []
    for index, (images, labels, paths) in enumerate(test_loader):
        image, image_path = images.to(DEVICE), paths[0]
        heatmap = heatmap_cache[image_path]["heatmap"]
        target_class = heatmap_cache[image_path]["predicted_class"]
        original_confidence = get_model_confidence(model, image, target_class)
        for ratio in DELETION_RATIOS:
            deleted = delete_top_heatmap_regions(image, heatmap, ratio)
            deleted_confidence = get_model_confidence(model, deleted, target_class)
            fidelity_records.append({
                "seed": model_seed,
                "index": index,
                "image_path": image_path,
                "true_label": int(labels.item()),
                "target_label": target_class,
                "deletion_ratio": ratio,
                "original_confidence": original_confidence,
                "deleted_confidence": deleted_confidence,
                "confidence_drop": original_confidence - deleted_confidence,
            })
        if (index + 1) % 25 == 0 or index + 1 == len(test_loader):
            print(f"Fidelity: {index + 1}/{len(test_loader)}")

    heatmap_df = pd.DataFrame(heatmap_records)
    consistency_df = pd.DataFrame(consistency_records)
    plausibility_df = pd.DataFrame(plausibility_records)
    fidelity_df = pd.DataFrame(fidelity_records)
    heatmap_df.to_csv(run_dir / "heatmap_records.csv", index=False)
    consistency_df.to_csv(run_dir / "consistency_results.csv", index=False)
    plausibility_df.to_csv(run_dir / "plausibility_results.csv", index=False)
    fidelity_df.to_csv(run_dir / "fidelity_deletion_results.csv", index=False)

    valid_masks = plausibility_df.loc[plausibility_df["mask_found"]]
    fidelity_90 = fidelity_df.loc[np.isclose(fidelity_df["deletion_ratio"], 0.90)]
    summary = {
        "seed": model_seed,
        "model_path": str(model_path),
        "number_of_test_images": len(heatmap_df),
        "number_of_masks": len(valid_masks),
        "consistency_ssim": consistency_df["ssim"].mean(),
        "consistency_mse": consistency_df["mse"].mean(),
        "consistency_iou": consistency_df["iou"].mean(),
        "plausibility_iou": valid_masks["gradcam_mask_iou"].mean(),
        "fidelity_overall_confidence_drop": fidelity_df["confidence_drop"].mean(),
        "fidelity_drop_at_90_percent": fidelity_90["confidence_drop"].mean(),
        "deleted_confidence_at_90_percent": fidelity_90["deleted_confidence"].mean(),
    }
    gradcam.remove_hooks()
    del gradcam, model, heatmap_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, consistency_df, plausibility_df, fidelity_df

In [12]:
# Run all the 10 checkpoints and summarize the between-run variability
def main():
    missing = [(seed, path) for seed, path in MODEL_PATHS.items() if not path.exists()]
    if missing:
        lines = "\n".join(f"  Seed {seed}: {path}" for seed, path in missing)
        raise FileNotFoundError(
            "Update MODEL_PATHS near the top of this file. These checkpoints were not found:\n" + lines
        )
    if len(MODEL_PATHS) != 10 or len(set(MODEL_PATHS)) != 10:
        raise ValueError("MODEL_PATHS must contain exactly 10 unique seeds.")

    class_names, _, test_samples, test_loader = prepare_test_data()
    summaries, consistency_parts, plausibility_parts, fidelity_parts = [], [], [], [],
    for run_number, (seed, path) in enumerate(MODEL_PATHS.items(), start=1):
        print(f"\nRUN {run_number}/10")
        summary, consistency, plausibility, fidelity = evaluate_checkpoint(
            seed, path, len(class_names), test_loader
        )
        summaries.append(summary)
        consistency_parts.append(consistency)
        plausibility_parts.append(plausibility)
        fidelity_parts.append(fidelity)

    per_run = pd.DataFrame(summaries)
    metrics = [
        "consistency_ssim",
        "consistency_mse",
        "consistency_iou",
        "plausibility_iou",
        "fidelity_overall_confidence_drop",
        "fidelity_drop_at_90_percent",
        "deleted_confidence_at_90_percent",
    ]
    mean_sd_rows = []
    for metric in metrics:
        values = per_run[metric].dropna().astype(float)
        mean, sd = values.mean(), values.std(ddof=1)
        mean_sd_rows.append({
            "metric": metric,
            "mean": mean,
            "sd": sd,
            "mean_sd": f"{mean:.4f} ± {sd:.4f}",
            "number_of_runs": len(values),
        })
    mean_sd = pd.DataFrame(mean_sd_rows)

    all_consistency = pd.concat(consistency_parts, ignore_index=True)
    all_plausibility = pd.concat(plausibility_parts, ignore_index=True)
    all_fidelity = pd.concat(fidelity_parts, ignore_index=True)
    per_run.to_csv(OUTPUT_DIR / "gradcampp_xai_per_run_results.csv", index=False)
    mean_sd.to_csv(OUTPUT_DIR / "gradcampp_xai_mean_sd_10_runs.csv", index=False)
    all_consistency.to_csv(OUTPUT_DIR / "gradcampp_all_consistency_results.csv", index=False)
    all_plausibility.to_csv(OUTPUT_DIR / "gradcampp_all_plausibility_results.csv", index=False)
    all_fidelity.to_csv(OUTPUT_DIR / "gradcampp_all_fidelity_results.csv", index=False)

    ratio_summary = all_fidelity.groupby(["seed", "deletion_ratio"], as_index=False).agg(
        mean_deleted_confidence=("deleted_confidence", "mean"),
        mean_confidence_drop=("confidence_drop", "mean"),
    )
    curve = ratio_summary.groupby("deletion_ratio", as_index=False).agg(
        mean_deleted_confidence=("mean_deleted_confidence", "mean"),
        sd_deleted_confidence=("mean_deleted_confidence", "std"),
        mean_confidence_drop=("mean_confidence_drop", "mean"),
        sd_confidence_drop=("mean_confidence_drop", "std"),
    )
    curve.to_csv(OUTPUT_DIR / "gradcampp_deletion_curve_mean_sd.csv", index=False)
    plt.figure(figsize=(7, 5))
    plt.errorbar(
        curve["deletion_ratio"], curve["mean_deleted_confidence"],
        yerr=curve["sd_deleted_confidence"], marker="o", capsize=4, linewidth=2,
    )
    plt.xlabel("Deletion Ratio")
    plt.ylabel("Mean Prediction Confidence")
    plt.title("Grad-CAM++ Deletion Test Across 10 Runs")
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "gradcampp_deletion_curve_10_runs.png", dpi=300)
    plt.close()

    metadata = {
        "evaluation_seed": EVALUATION_SEED,
        "model_seeds": list(MODEL_PATHS),
        "number_of_test_images": len(test_samples),
        "heatmap_threshold": HEATMAP_THRESHOLD,
        "consistency_perturbations": CONSISTENCY_PERTURBATIONS,
        "deletion_ratios": DELETION_RATIOS,
    }
    with open(OUTPUT_DIR / "evaluation_configuration.json", "w") as file:
        json.dump(metadata, file, indent=2)

    print("\n" + "=" * 72)
    print("FINAL XAI MEAN ± SD ACROSS 10 TRAINED MODELS")
    print("=" * 72)
    print(mean_sd.to_string(index=False))
    print("\nResults saved to:", OUTPUT_DIR)


if __name__ == "__main__":
    main()

Device: cpu
Classes: ['Adenocarcinoma', 'High-grade IN', 'Low-grade IN', 'Normal', 'Polyp', 'Serrated adenoma']
Total/train/validation/test: 2228 1559 334 335

RUN 1/10

MODEL SEED 42: outputs/resnet34/run_cache/run_0_seed_42_model.pth
Heatmaps: 25/335
Heatmaps: 50/335
Heatmaps: 75/335
Heatmaps: 100/335
Heatmaps: 125/335
Heatmaps: 150/335
Heatmaps: 175/335
Heatmaps: 200/335
Heatmaps: 225/335
Heatmaps: 250/335
Heatmaps: 275/335
Heatmaps: 300/335
Heatmaps: 325/335
Heatmaps: 335/335
Consistency: 25/335
Consistency: 50/335
Consistency: 75/335
Consistency: 100/335
Consistency: 125/335
Consistency: 150/335
Consistency: 175/335
Consistency: 200/335
Consistency: 225/335
Consistency: 250/335
Consistency: 275/335
Consistency: 300/335
Consistency: 325/335
Consistency: 335/335
Fidelity: 25/335
Fidelity: 50/335
Fidelity: 75/335
Fidelity: 100/335
Fidelity: 125/335
Fidelity: 150/335
Fidelity: 175/335
Fidelity: 200/335
Fidelity: 225/335
Fidelity: 250/335
Fidelity: 275/335
Fidelity: 300/335
Fidelity: 3

## Consistency

In [13]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("outputs/xai_evaluation_10_runs")
consistency_df = pd.read_csv(OUTPUT_DIR / "gradcampp_all_consistency_results.csv")
required_columns = {"seed", "perturbation", "ssim", "mse", "iou"}
missing_columns = required_columns - set(consistency_df.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

# Check that all 10 checkpoints are included
number_of_seeds = consistency_df["seed"].nunique()

if number_of_seeds != 10:
    raise ValueError(f"Expected 10 seeds, but found {number_of_seeds}.")

# Step 1: calculate one mean per seed for each perturbation
per_seed_perturbation = (
    consistency_df
    .groupby(["seed", "perturbation"], as_index=False)
    .agg(
        ssim=("ssim", "mean"),
        mse=("mse", "mean"),
        iou=("iou", "mean")
    )
)

# Step 2: calculate mean ± sample SD across the 10 checkpoints
perturbation_summary = (
    per_seed_perturbation
    .groupby("perturbation", as_index=False)
    .agg(
        ssim_mean=("ssim", "mean"),
        ssim_sd=("ssim", "std"),
        mse_mean=("mse", "mean"),
        mse_sd=("mse", "std"),
        iou_mean=("iou", "mean"),
        iou_sd=("iou", "std"),
        number_of_runs=("seed", "nunique")
    )
)

perturbation_summary["SSIM (mean ± SD)"] = (
    perturbation_summary["ssim_mean"].map("{:.4f}".format)
    + " ± "
    + perturbation_summary["ssim_sd"].map("{:.4f}".format)
)

perturbation_summary["MSE (mean ± SD)"] = (
    perturbation_summary["mse_mean"].map("{:.4f}".format)
    + " ± "
    + perturbation_summary["mse_sd"].map("{:.4f}".format)
)

perturbation_summary["IoU (mean ± SD)"] = (
    perturbation_summary["iou_mean"].map("{:.4f}".format)
    + " ± "
    + perturbation_summary["iou_sd"].map("{:.4f}".format)
)

display_columns = [
    "perturbation",
    "SSIM (mean ± SD)",
    "MSE (mean ± SD)",
    "IoU (mean ± SD)",
    "number_of_runs"
]

## Plausibility

In [14]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("outputs/xai_evaluation_10_runs")
plausibility_df = pd.read_csv(OUTPUT_DIR / "gradcampp_all_plausibility_results.csv")

# Recover class name from saved image path
plausibility_df["Class"] = plausibility_df["image_path"].apply(
    lambda x: Path(x).parent.name)

print("=== CLASS MAPPING RECOVERED FROM SAVED CSV ===")
class_mapping = (
    plausibility_df[["true_label", "Class"]]
    .drop_duplicates()
    .sort_values("true_label")
)
print(class_mapping.to_string(index=False))


# Overall plausibility IoU per run
overall_per_run = (
    plausibility_df
    .groupby("seed")["gradcam_mask_iou"]
    .mean()
)
print("\n=== OVERALL PLAUSIBILITY IoU BY RUN ===")
print(overall_per_run.to_string())


# Overall mean ± SD across 10 runs
overall_mean = overall_per_run.mean()
overall_sd = overall_per_run.std(ddof=1)
print("\n=== OVERALL PLAUSIBILITY IoU ===")
print(f"{overall_mean:.4f} ± {overall_sd:.4f}")

# Per-class IoU within each run
class_per_run = (
    plausibility_df
    .groupby(["seed", "true_label", "Class"])["gradcam_mask_iou"]
    .mean()
    .reset_index()
)
print("\n=== PER-CLASS PLAUSIBILITY IoU BY RUN ===")
print(class_per_run.to_string(index=False))

# Per-class mean ± SD across 10 runs
class_summary = (
    class_per_run
    .groupby(["true_label", "Class"])["gradcam_mask_iou"]
    .agg(mean="mean", std="std")
    .reset_index()
)
class_summary["Mean ± SD"] = class_summary.apply(
    lambda row:
    f"{row['mean']:.4f} ± {row['std']:.4f}",
    axis=1
)
print("\n=== PER-CLASS PLAUSIBILITY SUMMARY ===")
print(
    class_summary[
        ["true_label", "Class",
         "mean", "std", "Mean ± SD"]
    ].to_string(index=False)
)

# Consolidate into a table
publication_table = class_summary[["Class", "Mean ± SD"]].copy()
publication_table.columns = ["Class", "Plausibility IoU (Mean ± SD)"]
overall_row = pd.DataFrame({
    "Class": ["Overall"],
    "Plausibility IoU (Mean ± SD)": [f"{overall_mean:.4f} ± {overall_sd:.4f}"]
})
publication_table = pd.concat([publication_table, overall_row],  ignore_index=True)
print("\n=== FINAL PLAUSIBILITY RESULTS TABLE ===")
print(publication_table.to_string(index=False))

=== CLASS MAPPING RECOVERED FROM SAVED CSV ===
 true_label Class
          0 image
          1 image
          2 image
          3 image
          4 image
          5 image

=== OVERALL PLAUSIBILITY IoU BY RUN ===
seed
7       0.427822
42      0.449673
77      0.419597
123     0.441250
314     0.436727
2024    0.427232
2025    0.430265
3407    0.438636
8888    0.432564
9999    0.424982

=== OVERALL PLAUSIBILITY IoU ===
0.4329 ± 0.0088

=== PER-CLASS PLAUSIBILITY IoU BY RUN ===
 seed  true_label Class  gradcam_mask_iou
    7           0 image          0.403097
    7           1 image          0.472881
    7           2 image          0.492953
    7           3 image          0.400138
    7           4 image          0.371298
    7           5 image          0.402615
   42           0 image          0.446729
   42           1 image          0.454073
   42           2 image          0.505881
   42           3 image          0.395302
   42           4 image          0.394240
   42         

## Fidelity

In [15]:
import pandas as pd
import numpy as np

curve_path = (
    "outputs/xai_evaluation_10_runs/"
    "gradcampp_deletion_curve_mean_sd.csv"
)

curve_df = pd.read_csv(curve_path)

selected_ratios = curve_df[
    np.isclose(curve_df["deletion_ratio"], 0.10) |
    np.isclose(curve_df["deletion_ratio"], 0.20) |
    np.isclose(curve_df["deletion_ratio"], 0.30) |
    np.isclose(curve_df["deletion_ratio"], 0.40) |
    np.isclose(curve_df["deletion_ratio"], 0.50) |
    np.isclose(curve_df["deletion_ratio"], 0.60) |
    np.isclose(curve_df["deletion_ratio"], 0.70) |
    np.isclose(curve_df["deletion_ratio"], 0.80) |
    np.isclose(curve_df["deletion_ratio"], 0.90)
].copy()

selected_ratios["Confidence Drop (Mean ± SD)"] = selected_ratios.apply(
    lambda row: (
        f"{row['mean_confidence_drop']:.4f} ± "
        f"{row['sd_confidence_drop']:.4f}"
    ),
    axis=1
)

selected_ratios["Deletion Percentage"] = (
    selected_ratios["deletion_ratio"] * 100
).astype(int).astype(str) + "%"

result_table = selected_ratios[
    [
        "Deletion Percentage",
        "mean_confidence_drop",
        "sd_confidence_drop",
        "Confidence Drop (Mean ± SD)"
    ]
]

print(result_table.to_string(index=False))

Deletion Percentage  mean_confidence_drop  sd_confidence_drop Confidence Drop (Mean ± SD)
                10%              0.118584            0.012115             0.1186 ± 0.0121
                20%              0.215325            0.018253             0.2153 ± 0.0183
                30%              0.293917            0.022758             0.2939 ± 0.0228
                40%              0.358653            0.024994             0.3587 ± 0.0250
                50%              0.411928            0.026402             0.4119 ± 0.0264
                60%              0.457643            0.022519             0.4576 ± 0.0225
                70%              0.495691            0.020303             0.4957 ± 0.0203
                80%              0.528318            0.017823             0.5283 ± 0.0178
                90%              0.571351            0.021030             0.5714 ± 0.0210
